# v7_long_runs — resumable local Neural-LV search

This notebook is the **local-compute companion** to the scoped NLV v7 search.

Run the two priority branches **sequentially** on your machine:

1. `robust_mse` — resumes after the already-completed 2024-03-04 date.
2. `robust_huber3` — resumes after its already-completed 2024-03-04 date.

The recovered March-4 work is reused. New training is checkpointed **mid-date every 5 SDE updates**, including the model, hedge network, Adam states, NumPy sampler state, PyTorch RNG state, monitor history and candidate states. If VS Code or Python is interrupted, rerunning the same cell resumes from the last mid-date checkpoint.

**Do not run `robust_mse8k`, `robust_twobatch`, `robust_twostage`, or the later-March frozen test yet.** Those are deliberately left for the next decision after the two priority development branches finish.

Expected local time, based on the recovered runs:
- `robust_mse`: March 4 required about 78 minutes at one PyTorch thread. Three remaining dates are likely several hours.
- `robust_huber3`: March 4 required about 49 minutes. Three remaining dates are also likely several hours.
- This notebook uses **3 PyTorch CPU threads**, which previously gave a material speed-up on your machine. Run only one branch at a time.


**Important:** keep `v7_long_run_workspace/` after starting. The setup cell detects it and will not re-extract the old recovery cache over your new progress. If you deliberately want a clean restart, rename or delete that workspace yourself.


In [1]:
from pathlib import Path
import os, sys, zipfile, shutil, subprocess, textwrap, json, time

# Put this notebook and the two recovery ZIPs in the same folder.
ROOT = Path.cwd()
SCRIPTS_ZIP = ROOT / "neural_lv_v7_experiment_scripts_recovery.zip"
CACHE_ZIP = ROOT / "neural_lv_v7_optimization_cache_results.zip"
WORK = ROOT / "v7_long_run_workspace"

print("ROOT:", ROOT)
print("Scripts ZIP exists:", SCRIPTS_ZIP.exists())
print("Cache ZIP exists:", CACHE_ZIP.exists())

if not SCRIPTS_ZIP.exists() or not CACHE_ZIP.exists():
    raise FileNotFoundError(
        "Place neural_lv_v7_experiment_scripts_recovery.zip and "
        "neural_lv_v7_optimization_cache_results.zip beside this notebook."
    )


ROOT: c:\Users\Startklar\ICL\NDEs\Codes\NSDE\GPT_NSDE_work
Scripts ZIP exists: True
Cache ZIP exists: True


In [2]:

WORK.mkdir(exist_ok=True)
INIT_MARKER = WORK / ".v7_long_runs_initialized"

# IMPORTANT: unpack the recovery archives only on the first run.
# Re-extracting the old cache after a long run could overwrite updated partial CSVs.
if not INIT_MARKER.exists():
    print("Initialising long-run workspace from recovery bundles...")
    with zipfile.ZipFile(SCRIPTS_ZIP, "r") as z:
        z.extractall(WORK)
    with zipfile.ZipFile(CACHE_ZIP, "r") as z:
        z.extractall(WORK)
    INIT_MARKER.write_text("initialized\n", encoding="utf-8")
else:
    print("Existing workspace detected; recovery archives will NOT be re-extracted.")

# audit_v7.py in the recovery bundle referred to the original chat sandbox path.
# Point it at the notebook bundled with the scripts. Safe to repeat.
audit_path = WORK / "audit_v7.py"
audit_text = audit_path.read_text(encoding="utf-8")
audit_text = audit_text.replace(
    'NOTEBOOK = ROOT.parent / "upload" / "05_Neural_LV_v7_final_revision_crash_safe(1)(1).ipynb"',
    'NOTEBOOK = ROOT / "05_Neural_LV_v7_final_revision_crash_safe.ipynb"'
)
audit_path.write_text(audit_text, encoding="utf-8")

runner_path = WORK / "resumable_long_runner.py"
runner_path.write_text('\nfrom __future__ import annotations\n\nimport argparse\nimport gc\nimport json\nimport os\nimport time\nfrom dataclasses import replace\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\n\nimport search_pipeline as sp\nfrom audit_v7 import load_notebook_environment\n\n\nSAVE_EVERY_UPDATES = 5\nNUM_THREADS = 3\n\n\ndef atomic_torch_save(payload, path: Path):\n    path = Path(path)\n    tmp = path.with_suffix(path.suffix + ".tmp")\n    torch.save(payload, tmp)\n    os.replace(tmp, path)\n\n\ndef load_finished_model(ns, checkpoint_path: Path):\n    payload = torch.load(checkpoint_path, map_location="cpu", weights_only=False)\n    model = ns["MaturityHeadNeuralLocalVol"](payload["boundaries"]).cpu()\n    model.load_state_dict(payload["final_model_state"], strict=True)\n    model.eval()\n    sp.set_grad(model, False)\n    return model\n\n\ndef _save_middate(progress_path, cfg, date, previous_date, model, hedge,\n                  opt_model, opt_hedge, sampler, history, candidates,\n                  best, best_update, bad, last_update, elapsed_so_far):\n    payload = {\n        "version": "v7_long_runs_middate_v1",\n        "config": sp.asdict(cfg),\n        "date": str(pd.Timestamp(date).date()),\n        "previous_date": str(pd.Timestamp(previous_date).date()),\n        "model_state": sp.cpu_state_dict(model),\n        "hedge_state": {k: v.detach().cpu().clone() for k, v in hedge.state_dict().items()},\n        "opt_model": opt_model.state_dict(),\n        "opt_hedge": opt_hedge.state_dict(),\n        "sampler_rng_state": sampler.rng.bit_generator.state,\n        "torch_rng_state": torch.get_rng_state(),\n        "history": history,\n        "candidates": candidates,\n        "best": float(best),\n        "best_update": int(best_update),\n        "bad": int(bad),\n        "last_update": int(last_update),\n        "elapsed_so_far": float(elapsed_so_far),\n    }\n    atomic_torch_save(payload, progress_path)\n\n\ndef train_day_resumable(ns, cfg, date, previous_model, previous_date, output_dir,\n                        contracts=None, artifact_prefix=""):\n    selected, _, _ = ns["daily_split"](date)\n    training_contracts = selected if contracts is None else contracts\n    ctx = ns["build_calibration_context"](date, training_contracts)\n    model = ns["MaturityHeadNeuralLocalVol"](ctx.maturity_boundaries).cpu()\n    ns["transfer_rolling_model"](previous_model, model, previous_date, date)\n    hedge = ns["IntervalHedgeBank"](ctx.maturity_boundaries, len(ctx.contracts)).cpu()\n\n    opt_model = torch.optim.Adam(model.parameters(), lr=cfg.sde_lr)\n    opt_hedge = torch.optim.Adam(hedge.parameters(), lr=cfg.hedge_lr)\n    sampler = ns["UniformContractSampler"](\n        ctx, cfg.contracts_per_step,\n        ns["SEED"] + cfg.seed_offset + int(pd.Timestamp(date).day) * 101,\n    )\n\n    monitor_seed = ns["SEED"] + cfg.seed_offset + int(pd.Timestamp(date).day) * 1009\n    z_monitor = ns["antithetic_normals"](\n        cfg.monitor_paths, len(ctx.grid_np) - 1, monitor_seed\n    )\n\n    stamp = f"{artifact_prefix}{pd.Timestamp(date):%Y%m%d}"\n    progress_path = output_dir / f"{stamp}_inprogress.pt"\n\n    history = []\n    candidates = []\n\n    def record(update, train_value=np.nan):\n        model.eval()\n        sp.set_grad(model, False)\n        diagnostic = ns["objective_diagnostics"](model, ctx, z_monitor)\n        score = float(diagnostic["raw_rmse"])\n        candidates.append({\n            "update": int(update),\n            "monitor_rmse": score,\n            "state": sp.cpu_state_dict(model),\n        })\n        history.append({\n            "update": int(update),\n            "train_loss": float(train_value),\n            "monitor_rmse": score,\n        })\n        return score\n\n    start_time = time.time()\n    elapsed_before = 0.0\n    start_update = 0\n\n    if progress_path.exists():\n        p = torch.load(progress_path, map_location="cpu", weights_only=False)\n        if p.get("config", {}).get("name") != cfg.name:\n            raise RuntimeError(f"Config mismatch in {progress_path}")\n        model.load_state_dict(p["model_state"], strict=True)\n        hedge.load_state_dict(p["hedge_state"], strict=True)\n        opt_model.load_state_dict(p["opt_model"])\n        opt_hedge.load_state_dict(p["opt_hedge"])\n        sampler.rng.bit_generator.state = p["sampler_rng_state"]\n        torch.set_rng_state(p["torch_rng_state"])\n        history = list(p["history"])\n        candidates = list(p["candidates"])\n        best = float(p["best"])\n        best_update = int(p["best_update"])\n        bad = int(p["bad"])\n        start_update = int(p["last_update"])\n        elapsed_before = float(p.get("elapsed_so_far", 0.0))\n        print(\n            f"[resume] {cfg.name} {pd.Timestamp(date).date()} "\n            f"from completed update {start_update}",\n            flush=True,\n        )\n        del p\n        gc.collect()\n    else:\n        torch.manual_seed(ns["SEED"] + cfg.seed_offset + int(pd.Timestamp(date).day) * 17)\n        best = record(0)\n        best_update = 0\n        bad = 0\n\n    for update in range(start_update + 1, cfg.max_sde_updates + 1):\n        model.train()\n        hedge.eval()\n        sp.set_grad(model, True)\n        sp.set_grad(hedge, False)\n        model_losses = []\n\n        for _ in range(cfg.steps_per_phase):\n            ids = sampler.draw()\n            opt_model.zero_grad(set_to_none=True)\n            loss, scratch = sp.training_loss(ns, model, hedge, ctx, ids, cfg.objective, cfg)\n            loss.backward()\n            nn.utils.clip_grad_norm_(model.parameters(), 5.0)\n            opt_model.step()\n            model_losses.append(float(loss.detach()))\n            del loss, scratch\n\n        score = record(update, np.mean(model_losses))\n        if score < best:\n            best, best_update, bad = score, update, 0\n        else:\n            bad += 1\n\n        stop_now = update >= cfg.early_min_sde and bad >= cfg.early_patience_sde\n\n        # Match original search protocol: skip hedge phase if early stop has fired.\n        if not stop_now:\n            model.eval()\n            hedge.train()\n            sp.set_grad(model, False)\n            sp.set_grad(hedge, True)\n            for _ in range(cfg.steps_per_phase):\n                ids = sampler.draw()\n                z = torch.randn(\n                    cfg.paths_per_step, len(ctx.grid_np) - 1, dtype=torch.float32\n                )\n                opt_hedge.zero_grad(set_to_none=True)\n                _, cv = ns["simulate_estimators"](model, hedge, ctx, ids, z, "hedge")\n                variance = torch.var(cv / ctx.spot, dim=0, unbiased=False).mean()\n                variance.backward()\n                nn.utils.clip_grad_norm_(hedge.parameters(), 3.0)\n                opt_hedge.step()\n                del z, cv, variance\n\n        elapsed = elapsed_before + (time.time() - start_time)\n        if (\n            update % SAVE_EVERY_UPDATES == 0\n            or stop_now\n            or update == cfg.max_sde_updates\n        ):\n            _save_middate(\n                progress_path, cfg, date, previous_date,\n                model, hedge, opt_model, opt_hedge, sampler,\n                history, candidates, best, best_update, bad, update, elapsed,\n            )\n            print(\n                f"[checkpoint] {cfg.name} {pd.Timestamp(date).date()} "\n                f"update={update} monitor={score:.4f} best={best:.4f}",\n                flush=True,\n            )\n\n        if stop_now:\n            print(\n                f"[early stop] {cfg.name} {pd.Timestamp(date).date()} "\n                f"at update {update}; best monitor update {best_update}",\n                flush=True,\n            )\n            break\n\n        gc.collect()\n\n    del z_monitor, opt_model, opt_hedge, hedge\n    gc.collect()\n\n    history_frame = pd.DataFrame(history)\n\n    # Same robust checkpoint selection as the recovered search.\n    monitor_ids = set(\n        history_frame.nsmallest(cfg.screen_top_k, "monitor_rmse")["update"].astype(int)\n    )\n    monitor_ids.update(\n        history_frame.loc[history_frame["update"] % 5 == 0, "update"].astype(int)\n    )\n    monitor_ids.add(best_update)\n    screened = [c for c in candidates if c["update"] in monitor_ids]\n\n    screen, screen_seed_rows = sp.robust_scores(\n        ns, ctx, screened, cfg.screen_paths, cfg.screen_seeds,\n        ns["SEED"] + cfg.seed_offset + 700_000 + int(pd.Timestamp(date).day) * 100,\n    )\n    finalist_ids = set(screen.head(cfg.final_top_k)["update"].astype(int))\n    finalists = [c for c in screened if c["update"] in finalist_ids]\n\n    final, final_seed_rows = sp.robust_scores(\n        ns, ctx, finalists, cfg.final_paths, cfg.final_seeds,\n        ns["SEED"] + cfg.seed_offset + 900_000 + int(pd.Timestamp(date).day) * 100,\n    )\n\n    chosen_update = int(final.iloc[0]["update"])\n    chosen = next(c for c in finalists if c["update"] == chosen_update)\n    model.load_state_dict(chosen["state"], strict=True)\n    model.eval()\n    sp.set_grad(model, False)\n\n    history_frame.to_csv(output_dir / f"{stamp}_history.csv", index=False)\n    screen.to_csv(output_dir / f"{stamp}_screen.csv", index=False)\n    screen_seed_rows.to_csv(output_dir / f"{stamp}_screen_seeds.csv", index=False)\n    final.to_csv(output_dir / f"{stamp}_final_rerank.csv", index=False)\n    final_seed_rows.to_csv(output_dir / f"{stamp}_final_rerank_seeds.csv", index=False)\n\n    torch.save({\n        "config": sp.asdict(cfg),\n        "date": str(pd.Timestamp(date).date()),\n        "previous_date": str(pd.Timestamp(previous_date).date()),\n        "boundaries": ctx.maturity_boundaries,\n        "chosen_update": chosen_update,\n        "final_model_state": sp.cpu_state_dict(model),\n        "candidate_states": candidates,\n    }, output_dir / f"{stamp}_checkpoint.pt")\n\n    progress_path.unlink(missing_ok=True)\n\n    meta = {\n        "date": pd.Timestamp(date),\n        "updates_run": int(history_frame["update"].max()),\n        "monitor_best_update": int(best_update),\n        "chosen_update": chosen_update,\n        "chosen_rerank_rmse": float(final.iloc[0].aggregate_rmse),\n        "elapsed_seconds": elapsed_before + (time.time() - start_time),\n        "stage": artifact_prefix.rstrip("_") or "forecast_full",\n    }\n    del candidates, screened, finalists\n    gc.collect()\n    return model, ctx, meta\n\n\ndef _existing_row(path: Path, date: pd.Timestamp):\n    if not path.exists():\n        return None\n    frame = pd.read_csv(path)\n    if "date" not in frame.columns:\n        return None\n    frame["date"] = pd.to_datetime(frame["date"])\n    hit = frame.loc[frame["date"] == pd.Timestamp(date)]\n    return None if hit.empty else hit.iloc[-1].to_dict()\n\n\ndef run_development_resumable(variant):\n    cfg = sp.VARIANTS[variant]\n    output_dir = sp.RUN_ROOT / variant\n    output_dir.mkdir(exist_ok=True)\n    (output_dir / "config.json").write_text(\n        json.dumps(sp.asdict(cfg), indent=2), encoding="utf-8"\n    )\n\n    ns = load_notebook_environment()\n    ns["stage_log"] = lambda message: None\n    ns["FINAL_PATH_CHUNK"] = 2000\n    torch.set_num_threads(NUM_THREADS)\n\n    previous = sp.load_anchor(ns)\n    previous_date = pd.Timestamp("2024-03-01")\n\n    dates = [pd.Timestamp(x) for x in [\n        "2024-03-04", "2024-03-05", "2024-03-06", "2024-03-07"\n    ]]\n    next_dates = [pd.Timestamp(x) for x in [\n        "2024-03-05", "2024-03-06", "2024-03-07", "2024-03-08"\n    ]]\n\n    partial_summary_path = output_dir / "development_summary_partial.csv"\n    partial_meta_path = output_dir / "training_metadata_partial.csv"\n    summaries, metadata = [], []\n\n    for index, (date, next_date) in enumerate(zip(dates, next_dates), start=1):\n        checkpoint = output_dir / f"{date:%Y%m%d}_checkpoint.pt"\n        transition = output_dir / f"transition_{date:%Y%m%d}.csv"\n\n        if checkpoint.exists() and transition.exists():\n            print(f"[skip completed] {variant} {date.date()}", flush=True)\n            existing_s = _existing_row(partial_summary_path, date)\n            existing_m = _existing_row(partial_meta_path, date)\n            if existing_s is None or existing_m is None:\n                raise RuntimeError(\n                    f"Completed artifacts exist for {date.date()} but partial summary/meta is missing."\n                )\n            summaries.append(existing_s)\n            metadata.append(existing_m)\n            del previous\n            previous = load_finished_model(ns, checkpoint)\n            previous_date = date\n            gc.collect()\n            continue\n\n        print(\n            f"[{variant}] train {date.date()} -> evaluate {next_date.date()}",\n            flush=True,\n        )\n\n        if cfg.two_stage:\n            _, fit_contracts, _ = ns["daily_split"](date)\n            fit_model, fit_ctx, fit_meta = train_day_resumable(\n                ns, cfg, date, previous, previous_date, output_dir,\n                contracts=fit_contracts, artifact_prefix="fit80_",\n            )\n            polish_cfg = replace(\n                cfg,\n                two_stage=False,\n                max_sde_updates=cfg.polish_max_sde_updates,\n                early_min_sde=cfg.polish_early_min_sde,\n                early_patience_sde=cfg.polish_early_patience_sde,\n            )\n            model, ctx, meta = train_day_resumable(\n                ns, polish_cfg, date, fit_model, date, output_dir\n            )\n            meta.update({\n                "fit80_updates_run": fit_meta["updates_run"],\n                "fit80_monitor_best_update": fit_meta["monitor_best_update"],\n                "fit80_chosen_update": fit_meta["chosen_update"],\n                "fit80_chosen_rerank_rmse": fit_meta["chosen_rerank_rmse"],\n                "fit80_elapsed_seconds": fit_meta["elapsed_seconds"],\n            })\n            del fit_model, fit_ctx\n        else:\n            model, ctx, meta = train_day_resumable(\n                ns, cfg, date, previous, previous_date, output_dir\n            )\n\n        summary = sp.evaluate_one_transition(\n            ns, model, date, next_date, cfg, output_dir, index\n        )\n        summaries.append(summary)\n        metadata.append(meta)\n\n        pd.DataFrame(summaries).to_csv(partial_summary_path, index=False)\n        pd.DataFrame(metadata).to_csv(partial_meta_path, index=False)\n\n        del ctx, previous\n        previous, previous_date = model, date\n        gc.collect()\n\n    first = pd.read_csv(sp.BASE_CACHE / "v7_first_transition_summary.csv").iloc[0].to_dict()\n    combined = pd.concat(\n        [pd.DataFrame([first]), pd.DataFrame(summaries)], ignore_index=True\n    )\n    combined.to_csv(output_dir / "development_summary.csv", index=False)\n    pd.DataFrame(metadata).to_csv(output_dir / "training_metadata.csv", index=False)\n    print(f"[{variant}] complete", flush=True)\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--variant", choices=sorted(sp.VARIANTS), required=True)\n    args = parser.parse_args()\n    run_development_resumable(args.variant)\n\n\nif __name__ == "__main__":\n    main()\n', encoding="utf-8")

print("Workspace:", WORK)
print("Runner:", runner_path)
print("Recovered robust_mse Mar-4 checkpoint:",
      (WORK/"search_runs"/"robust_mse"/"20240304_checkpoint.pt").exists())
print("Recovered robust_huber3 Mar-4 checkpoint:",
      (WORK/"search_runs"/"robust_huber3"/"20240304_checkpoint.pt").exists())

# Static syntax check before any long run.
subprocess.run(
    [sys.executable, "-m", "py_compile",
     str(WORK/"search_pipeline.py"),
     str(WORK/"audit_v7.py"),
     str(runner_path)],
    check=True,
)
print("Syntax checks passed.")


Existing workspace detected; recovery archives will NOT be re-extracted.
Workspace: c:\Users\Startklar\ICL\NDEs\Codes\NSDE\GPT_NSDE_work\v7_long_run_workspace
Runner: c:\Users\Startklar\ICL\NDEs\Codes\NSDE\GPT_NSDE_work\v7_long_run_workspace\resumable_long_runner.py
Recovered robust_mse Mar-4 checkpoint: True
Recovered robust_huber3 Mar-4 checkpoint: True
Syntax checks passed.


In [3]:
from pathlib import Path

runner_path = WORK / "resumable_long_runner.py"
txt = runner_path.read_text(encoding="utf-8")

# Add tqdm.
if "from tqdm.auto import tqdm" not in txt:
    txt = txt.replace(
        "import torch.nn as nn",
        "import torch.nn as nn\nfrom tqdm.auto import tqdm"
    )

old_loop = """    for update in range(start_update + 1, cfg.max_sde_updates + 1):
"""

new_loop = """    update_bar = tqdm(
        range(start_update + 1, cfg.max_sde_updates + 1),
        total=cfg.max_sde_updates,
        initial=start_update,
        desc=f"{cfg.name} {pd.Timestamp(date):%Y-%m-%d}",
        unit="SDE",
        dynamic_ncols=True,
        mininterval=1.0,
    )

    for update in update_bar:
"""

if old_loop in txt:
    txt = txt.replace(old_loop, new_loop)

old_block = """        if score < best:
            best, best_update, bad = score, update, 0
        else:
            bad += 1

        stop_now = (
"""

new_block = """        if score < best:
            best, best_update, bad = score, update, 0
        else:
            bad += 1

        update_bar.set_postfix(
            raw_RMSE=f"{score:.3f}",
            best=f"{best:.3f}",
            patience=f"{bad}/{cfg.early_patience_sde}",
        )

        stop_now = (
"""

if old_block in txt:
    txt = txt.replace(old_block, new_block)

runner_path.write_text(txt, encoding="utf-8")
print("Added live SDE progress bars.")

Added live SDE progress bars.


## Status before running

This reports what is already complete and whether an interrupted mid-date checkpoint exists. Rerun it any time; it does not train anything.


In [4]:
import pandas as pd

for variant in ["robust_mse", "robust_huber3"]:
    d = WORK / "search_runs" / variant
    print("\n", variant)
    for day in ["20240304", "20240305", "20240306", "20240307"]:
        final_ckpt = d / f"{day}_checkpoint.pt"
        mid_ckpt = d / f"{day}_inprogress.pt"
        transition = d / f"transition_{day}.csv"
        print(
            day,
            "| final:", final_ckpt.exists(),
            "| transition:", transition.exists(),
            "| mid-date resume:", mid_ckpt.exists(),
        )



 robust_mse
20240304 | final: True | transition: True | mid-date resume: False
20240305 | final: False | transition: False | mid-date resume: True
20240306 | final: False | transition: False | mid-date resume: False
20240307 | final: False | transition: False | mid-date resume: False

 robust_huber3
20240304 | final: True | transition: True | mid-date resume: False
20240305 | final: False | transition: False | mid-date resume: False
20240306 | final: False | transition: False | mid-date resume: False
20240307 | final: False | transition: False | mid-date resume: False


In [5]:
import importlib.util
import sys

# Make the recovered search modules importable.
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))

runner_path = WORK / "resumable_long_runner.py"

spec = importlib.util.spec_from_file_location(
    "v7_long_runner", runner_path
)
v7runner = importlib.util.module_from_spec(spec)
spec.loader.exec_module(v7runner)

## Run 1 — `robust_mse` continuation

Start here. The runner should print `[skip completed] robust_mse 2024-03-04` and begin **2024-03-05**.

If the run is interrupted, simply rerun this cell. At most the work since the most recent 5-update checkpoint is lost.


In [6]:
# Runs directly in the notebook kernel, so prints/tqdm are visible live.
v7runner.run_development_resumable("robust_mse")

Device: cpu | PyTorch threads: 1
Compute profile: cpu | transitions: 5
Calibration universe: all filtered OTM
Objective: plain contract-level raw price MSE (spot-scaled)
Main budget: 100 SDE phases | polish: 30 SDE phases
LR: 0.0001 | hedge LR: 0.001 | scheduler: none
Early stop: 10 SDE phases without a new raw-RMSE best
options : one_month_spx_option_price.csv
forwards: one_month_spx_forward_price.csv
spot    : one_month_spx_security_price.csv
rates   : one_month_spx_zero_coupon_yield_curve.csv
Complete cleaned quotes: 192031
OTM calibration observations: 91930
Trading dates: 20
Expiry/settlement slices: 809
Anchor selected/train/validation: 4304 3443 861
Anchor selected as fraction of filtered OTM: 100.0%
Anchor maturities train/validation: 36 36
Anchor training contracts: 3443
Anchor maturity heads: 36
Anchor simulation steps: 129
[skip completed] robust_mse 2024-03-04
[robust_mse] train 2024-03-05 -> evaluate 2024-03-06
Warm-start 2024-03-04 -> 2024-03-05: 36 heads | elapsed=0.0027

robust_mse 2024-03-05:  25%|##5       | 25/100 [00:00<?, ?SDE/s]

[checkpoint] robust_mse 2024-03-05 update=30 monitor=1.8820 best=1.8820
[checkpoint] robust_mse 2024-03-05 update=35 monitor=1.8206 best=1.8206
[checkpoint] robust_mse 2024-03-05 update=40 monitor=1.8194 best=1.7896
[checkpoint] robust_mse 2024-03-05 update=45 monitor=1.7465 best=1.7465
[checkpoint] robust_mse 2024-03-05 update=50 monitor=2.3950 best=1.7272
[checkpoint] robust_mse 2024-03-05 update=55 monitor=1.8225 best=1.7272
[checkpoint] robust_mse 2024-03-05 update=60 monitor=1.9926 best=1.7035
[checkpoint] robust_mse 2024-03-05 update=65 monitor=2.2134 best=1.7035
[checkpoint] robust_mse 2024-03-05 update=70 monitor=1.8122 best=1.7035
[checkpoint] robust_mse 2024-03-05 update=75 monitor=2.5543 best=1.6888
[checkpoint] robust_mse 2024-03-05 update=80 monitor=2.7063 best=1.6888
[checkpoint] robust_mse 2024-03-05 update=85 monitor=1.6461 best=1.6461
[checkpoint] robust_mse 2024-03-05 update=90 monitor=2.0402 best=1.6461
[checkpoint] robust_mse 2024-03-05 update=95 monitor=2.4372 best

Future-date MC AM=1: 100%|██████████| 100000/100000 [00:17<00:00, 5632.03path/s]


[robust_mse] train 2024-03-06 -> evaluate 2024-03-07
Warm-start 2024-03-05 -> 2024-03-06: 36 heads | elapsed=0.00274y | median head mismatch=0.00000y


...

[robust_mse] train 2024-03-07 -> evaluate 2024-03-08
Warm-start 2024-03-06 -> 2024-03-07: 36 heads | elapsed=0.00274y | median head mismatch=0.00000y


robust_mse 2024-03-07:   0%|          | 0/100 [00:00<?, ?SDE/s]

[checkpoint] robust_mse 2024-03-07 update=5 monitor=2.7821 best=2.4487
[checkpoint] robust_mse 2024-03-07 update=10 monitor=2.2818 best=2.1148
[checkpoint] robust_mse 2024-03-07 update=15 monitor=2.0756 best=1.9079
[checkpoint] robust_mse 2024-03-07 update=20 monitor=2.7431 best=1.9079
[checkpoint] robust_mse 2024-03-07 update=25 monitor=2.5868 best=1.9079
[checkpoint] robust_mse 2024-03-07 update=30 monitor=2.7331 best=1.8297
[checkpoint] robust_mse 2024-03-07 update=35 monitor=1.9318 best=1.8297
[checkpoint] robust_mse 2024-03-07 update=40 monitor=2.3204 best=1.8297
[checkpoint] robust_mse 2024-03-07 update=45 monitor=1.7865 best=1.7865
[checkpoint] robust_mse 2024-03-07 update=50 monitor=1.8297 best=1.7865
[checkpoint] robust_mse 2024-03-07 update=55 monitor=1.8099 best=1.7783
[checkpoint] robust_mse 2024-03-07 update=60 monitor=1.8289 best=1.7783
[checkpoint] robust_mse 2024-03-07 update=65 monitor=2.0707 best=1.7597
[checkpoint] robust_mse 2024-03-07 update=70 monitor=2.1865 best=

Future-date MC AM=1: 100%|██████████| 100000/100000 [00:10<00:00, 9969.34path/s]


[robust_mse] complete


## Run 2 — `robust_huber3` continuation

Run this **after `robust_mse` finishes**. It likewise skips the completed March-4 boundary and starts on March 5.


In [9]:
v7runner.run_development_resumable("robust_huber3")

Device: cpu | PyTorch threads: 1
Compute profile: cpu | transitions: 5
Calibration universe: all filtered OTM
Objective: plain contract-level raw price MSE (spot-scaled)
Main budget: 100 SDE phases | polish: 30 SDE phases
LR: 0.0001 | hedge LR: 0.001 | scheduler: none
Early stop: 10 SDE phases without a new raw-RMSE best
options : one_month_spx_option_price.csv
forwards: one_month_spx_forward_price.csv
spot    : one_month_spx_security_price.csv
rates   : one_month_spx_zero_coupon_yield_curve.csv
Complete cleaned quotes: 192031
OTM calibration observations: 91930
Trading dates: 20
Expiry/settlement slices: 809
Anchor selected/train/validation: 4304 3443 861
Anchor selected as fraction of filtered OTM: 100.0%
Anchor maturities train/validation: 36 36
Anchor training contracts: 3443
Anchor maturity heads: 36
Anchor simulation steps: 129
[skip completed] robust_huber3 2024-03-04
[robust_huber3] train 2024-03-05 -> evaluate 2024-03-06
Warm-start 2024-03-04 -> 2024-03-05: 36 heads | elapsed=

robust_huber3 2024-03-05:   0%|          | 0/100 [00:00<?, ?SDE/s]

[checkpoint] robust_huber3 2024-03-05 update=5 monitor=2.8917 best=2.8917
[checkpoint] robust_huber3 2024-03-05 update=10 monitor=2.3899 best=2.3899
[checkpoint] robust_huber3 2024-03-05 update=15 monitor=3.0081 best=2.1905
[checkpoint] robust_huber3 2024-03-05 update=20 monitor=1.9894 best=1.9894
[checkpoint] robust_huber3 2024-03-05 update=25 monitor=1.8460 best=1.8460
[checkpoint] robust_huber3 2024-03-05 update=30 monitor=1.9271 best=1.8460
[checkpoint] robust_huber3 2024-03-05 update=35 monitor=1.7943 best=1.7943
[checkpoint] robust_huber3 2024-03-05 update=40 monitor=1.8479 best=1.7943
[checkpoint] robust_huber3 2024-03-05 update=45 monitor=1.8964 best=1.7943
[checkpoint] robust_huber3 2024-03-05 update=50 monitor=2.2234 best=1.7049
[checkpoint] robust_huber3 2024-03-05 update=55 monitor=1.9495 best=1.7049
[checkpoint] robust_huber3 2024-03-05 update=60 monitor=1.8015 best=1.6889
[checkpoint] robust_huber3 2024-03-05 update=65 monitor=2.5112 best=1.6889
[checkpoint] robust_huber3

Future-date MC AM=1: 100%|██████████| 100000/100000 [00:27<00:00, 3700.39path/s]


[robust_huber3] train 2024-03-06 -> evaluate 2024-03-07
Warm-start 2024-03-05 -> 2024-03-06: 36 heads | elapsed=0.00274y | median head mismatch=0.00000y


...

[robust_huber3] train 2024-03-07 -> evaluate 2024-03-08
Warm-start 2024-03-06 -> 2024-03-07: 36 heads | elapsed=0.00274y | median head mismatch=0.00000y


robust_huber3 2024-03-07:   0%|          | 0/100 [00:00<?, ?SDE/s]

[checkpoint] robust_huber3 2024-03-07 update=5 monitor=3.0069 best=2.8507
[checkpoint] robust_huber3 2024-03-07 update=10 monitor=2.2127 best=2.0723
[checkpoint] robust_huber3 2024-03-07 update=15 monitor=2.0243 best=1.9752
[checkpoint] robust_huber3 2024-03-07 update=20 monitor=2.8830 best=1.9467
[checkpoint] robust_huber3 2024-03-07 update=25 monitor=2.6544 best=1.9467
[checkpoint] robust_huber3 2024-03-07 update=30 monitor=2.7724 best=1.9467
[checkpoint] robust_huber3 2024-03-07 update=35 monitor=1.8431 best=1.8431
[checkpoint] robust_huber3 2024-03-07 update=40 monitor=2.5083 best=1.8154
[checkpoint] robust_huber3 2024-03-07 update=45 monitor=1.8382 best=1.8154
[checkpoint] robust_huber3 2024-03-07 update=50 monitor=2.1556 best=1.8154
[checkpoint] robust_huber3 2024-03-07 update=55 monitor=1.8448 best=1.7927
[checkpoint] robust_huber3 2024-03-07 update=60 monitor=1.9614 best=1.7927
[checkpoint] robust_huber3 2024-03-07 update=65 monitor=2.1771 best=1.7927
[checkpoint] robust_huber3

Future-date MC AM=1: 100%|██████████| 100000/100000 [00:10<00:00, 9158.65path/s]


[robust_huber3] complete


## Compact completion summary

Once both branches finish, this prints their five-transition development summaries. Do **not** use this cell to choose the final pipeline yourself; send the outputs back to me and I will combine them with the common-shock baseline and decide which remaining branch, if any, deserves another expensive run.


In [10]:
for variant in ["robust_mse", "robust_huber3"]:
    p = WORK / "search_runs" / variant / "development_summary.csv"
    print("\n===", variant, "===")
    if not p.exists():
        print("Not complete yet.")
        continue
    x = pd.read_csv(p)
    cols = [c for c in [
        "date", "next_date", "otm_n", "otm_pricing_RMSE",
        "otm_NLV_hedge_RMSE", "otm_market_delta_hedge_RMSE"
    ] if c in x.columns]
    print(x[cols].to_string(index=False))



=== robust_mse ===
               date           next_date  otm_n  otm_pricing_RMSE  otm_NLV_hedge_RMSE  otm_market_delta_hedge_RMSE
         2024-03-01          2024-03-04   3932          1.899206            0.807161                     0.763770
2024-03-04 00:00:00          2024-03-05   3951          4.851095            4.137969                     1.845664
2024-03-05 00:00:00 2024-03-06 00:00:00   4174          4.892166            0.475100                     1.337597
2024-03-06 00:00:00 2024-03-07 00:00:00   3917          6.535772            1.590811                     3.451301
2024-03-07 00:00:00 2024-03-08 00:00:00   4027          3.634192            1.578781                     0.764902

=== robust_huber3 ===
               date           next_date  otm_n  otm_pricing_RMSE  otm_NLV_hedge_RMSE  otm_market_delta_hedge_RMSE
         2024-03-01          2024-03-04   3932          1.899206            0.807161                     0.763770
2024-03-04 00:00:00          2024-03-05   395

## Package results to send back

Run this after the long branches finish—or after any interruption if you want me to inspect partial progress. It creates a small ZIP containing logs, histories, rerank tables, checkpoints, transition outputs and summaries for the two priority branches.


In [11]:
import zipfile

OUT_ZIP = ROOT / "v7_long_runs_results.zip"
include_dirs = [
    WORK / "search_runs" / "robust_mse",
    WORK / "search_runs" / "robust_huber3",
]
with zipfile.ZipFile(OUT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for folder in include_dirs:
        if not folder.exists():
            continue
        for p in folder.rglob("*"):
            if p.is_file():
                z.write(p, arcname=str(p.relative_to(WORK)))
print("Created:", OUT_ZIP)


Created: c:\Users\Startklar\ICL\NDEs\Codes\NSDE\GPT_NSDE_work\v7_long_runs_results.zip


## HOLD — additional long branches (ABANDONED)

Do not launch these yet.

After I inspect the completed MSE and Huber development paths, I will decide whether the remaining compute is best spent on:

- `robust_mse8k` — more MC paths per training step;
- `robust_twobatch` — unbiased two-independent-batch squared-loss estimator;
- `robust_twostage` — explicit 80% fit followed by 100% current-day polish;
- or no further development branch at all, followed directly by the **untouched later-March frozen test**.

This avoids spending another 3–6 hours on a variant whose motivation may already have been falsified by the two priority branches.


In [ ]:
# ONLY CHANGE THIS AFTER I tell you which additional branch is worth running.
OPTIONAL_VARIANT = None  # e.g. "robust_twobatch"

if OPTIONAL_VARIANT is not None:
    allowed = {"robust_mse8k", "robust_twobatch", "robust_twostage"}
    if OPTIONAL_VARIANT not in allowed:
        raise ValueError(f"Choose one of {sorted(allowed)}")
    cmd = [sys.executable, "-u", "resumable_long_runner.py", "--variant", OPTIONAL_VARIANT]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, cwd=WORK, check=True)
else:
    print("No optional long branch requested.")
